In [2]:
#----IMPORT LIBRAIRIES----
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns
import json

import pvlib

import mlflow
from mlflow.models.signature import infer_signature

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error

from sklearn.inspection import permutation_importance

import Model_func as mf
import boto3

from dotenv import load_dotenv
import os

load_dotenv()
os.environ["MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING"] = "false"


In [3]:
#---VARIABLES----
weather_data_path = 'https://renergies99-bucket.s3.eu-west-3.amazonaws.com/public/openweathermap/merge_openweathermap_cleaned.csv'
solar_data_path = 'https://renergies99-bucket.s3.eu-west-3.amazonaws.com/public/solar/raw_solar_data.csv'
landsat_data_path = 'https://renergies99-bucket.s3.eu-west-3.amazonaws.com/public/LandSat/result_EarthExplorer_region_ARA.csv'

prod_data_path = 'https://renergies99-bucket.s3.eu-west-3.amazonaws.com/public/prod/eCO2mix_RTE_Auvergne-Rhone-Alpes_cleaned.csv'
target = 'tch_solaire_(%)'


In [10]:
#--- PREPARATION ----
collected_weather_data = mf.data_collection_weather(weather_data_path) # collect data and format columns per city
collected_solar_data = mf.data_coll_solar(solar_data_path)
collected_landsat_data = mf.data_coll_landsat(landsat_data_path)
landsat_data = collected_landsat_data.copy()

weather_solar = mf.merge_weather_solar_data(collected_weather_data, collected_solar_data)

#creer un df landsat réduit avec 1 donnée/jour
columns_to_keep = landsat_data.select_dtypes(exclude=["object"]).columns
limited_landsat_data = landsat_data[columns_to_keep].groupby('Time').mean().reset_index()
 
merged_data = mf.merge_weather_solar_landsat_data(collected_weather_data, collected_solar_data, limited_landsat_data)

In [5]:
# merged_data_copy = pd.read_csv('../../../Mes_fichiers_vrac/merged_data_copy.csv')
# merged_data_copy['Time'] = pd.to_datetime(merged_data_copy['Time'])

# merged_data = merged_data_copy.copy()

In [11]:
#---data_split : add target and train_test_split
prod_data = mf.data_collection_prod(prod_data_path)
targeted_data = mf.add_target(merged_data, prod_data, target_columns_to_use=['Time', target])
#!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

#!!!! UNIQUEMENT LES DONNEES DE MOULINS ! !!!!!!

#!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
#---columns selection
col_solar = ['Ap', '10cm', 'K index Planetary']
col_weather = ['Moulins_temp', 'Moulins_feels_like' 'Moulins_pressure', 'Moulins_humidity', 'Moulins_dew_point',
                'Moulins_clouds', 'Moulins_wind_speed', 'Moulins_wind_deg']
features = col_weather + col_solar + ['Moulins_day_length'] + ['Moulins_Month']

selected_weather_columns = [col for col in targeted_data.columns if col.endswith(tuple(col_weather))]
selected_columns = selected_weather_columns + col_solar + ['Moulins_day_length'] + ['Moulins_Month']

y = targeted_data[target].to_numpy()
X = targeted_data[selected_columns]

#--- gestion de NaN
X = X.dropna(axis=1)
df = X.copy()

In [13]:
# Définition automatique des transformations à appliquer
transformations = {
    'Moulins_humidity': {'type': 'Low/High', 'seuil': 50},
    # 'temperature': {'type': 'Standardization', 'seuil': None},
    # 'pressure': {'type': 'Normalization', 'seuil': [100, 200]}
}

# Dictionnaire final pour stocker les infos de transfo
transform_info = {}

for col, info in transformations.items():
    if info['type'] == 'Low/High':
        # Création automatique des colonnes Low/High
        df[f'{col}_Low'] = df[col] * (df[col] < info['seuil'])
        df[f'{col}_High'] = df[col] * (df[col] >= info['seuil'])
        df = df.drop(col, axis=1)
    # autres types de transformation si besoin
    
    # Stockage des infos dans le dictionnaire
    transform_info[col] = {'type_transfo': info['type'], 'seuils': info['seuil']}

print(transform_info)
df.head()


{'Moulins_humidity': {'type_transfo': 'Low/High', 'seuils': 50}}


,Moulins_temp,Moulins_dew_point,Moulins_clouds,Moulins_wind_speed,Moulins_wind_deg,10cm,K index Planetary,Moulins_day_length,Moulins_Month,Moulins_humidity_Low,Moulins_humidity_High
0,1.11,-1.28,100,3.06,38,74.0,0.125,8.809722,1,0,83
1,0.71,-3.16,37,3.16,21,73.0,0.375,8.835833,1,0,73
2,-0.74,-3.56,70,1.46,291,73.0,2.125,8.863056,1,0,79
3,4.55,3.36,100,4.83,250,73.0,1.750,8.891389,1,0,92
4,8.50,8.05,100,4.19,278,73.0,0.375,8.921111,1,0,97


In [14]:
#---MLFlow params
os.environ["APP_URI"] = "https://renergies99-mlflow.hf.space/"
EXPERIMENT_NAME = "test_sur_MOULINS"

mlflow.set_tracking_uri(os.environ["APP_URI"])
mlflow.set_experiment(EXPERIMENT_NAME)
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

mlflow.sklearn.autolog()  # enables automatic logging for scikit-learn

#---Preprocess
result_preprocess = mf.preprocessing_and_pipeline(df)
pipeline = result_preprocess["pipeline"]
preprocessor = result_preprocess["preprocessor"]

run_description = (
    f"Features used: {features} \n Target: {target}\n"
    f"Estimator: {pipeline.named_steps['estimator']}\n"
    f"Transform_info : {transform_info}"
)

x_train, x_test, y_train, y_test = train_test_split(df, y, test_size=0.3, random_state=24)
input_example = x_train.iloc[:3]


with mlflow.start_run(experiment_id=experiment.experiment_id, description=run_description):
    # Fit the pipeline (preprocessing + model)
    pipeline.fit(x_train, y_train)

    # signature
    signature = infer_signature(x_test, pipeline.predict(x_test))

    # predictions
    y_pred = pipeline.predict(x_test)

    #Artifact for features_names
    #mf.custom_get_feature_names(result_preprocess, artifact_name="features.json")
 
    #artifact for confidence interval
    error = mf.error_stat(x_test, y_test, pipeline)
    error.to_json("error.json")
    mlflow.log_artifact("error.json")
    
    # metrics
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    n = len(y_test)
    p = X.shape[1]
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

    
    # logging metrics
    mlflow.log_metric("MAE", mae)
    mlflow.log_metric("MSE", mse)
    mlflow.log_metric("RMSE", rmse)
    mlflow.log_metric("R2", r2)
    mlflow.log_metric("Adjusted_R2", adj_r2)
 
    # Log the full pipeline as a model
    mlflow.sklearn.log_model(pipeline, signature=signature, input_example=input_example)



c:\Users\hardy\Documents\JEDHA\Jedha_DataScience_Fullstack\10_projet_final\REnergies_efficiency_prediction\code\02_Modeles\Model_func.py:416: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  table = df.groupby('quantile_group')['y_pred'].agg(['mean', 'std']).reset_index()


🏃 View run amusing-grub-887 at: https://renergies99-mlflow.hf.space/#/experiments/6/runs/bd2222e993404d6c9848b719f10623db
🧪 View experiment at: https://renergies99-mlflow.hf.space/#/experiments/6


# Results exploration
Coefficients

In [ ]:
preprocessor = pipeline.named_steps['preprocessor']

feature_names = []

for name, transformer, cols in preprocessor.transformers:
    if name == 'num':
        feature_names.extend(cols)  # StandardScaler ne change pas le nombre de colonnes
    elif name == 'obj':
        feature_names.extend(cols)  # passthrough garde les colonnes telles quelles

# Récupérer les coefficients du modèle
coefs = pipeline.named_steps['estimator'].coef_

print(len(feature_names), len(coefs))

# Tracer les coefs (triés en valuer absolue)
df_coef = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefs
})
df_coef = df_coef.sort_values(by='coefficient', key=abs)

px.bar(df_coef, x='coefficient', y='feature')

Erreur

In [15]:
# Calcul des prédictions et des résidus
y_pred = pipeline.predict(x_test)
residuals = y_test - y_pred

# Création de la figure
fig = go.Figure()

# Ajout des résidus
fig.add_trace(go.Scatter(
    x=y_pred,
    y=residuals,
    mode='markers',
    name='Résidus',
    marker=dict(color='blue', size=8)
))

# Ligne horizontale y=0
fig.add_trace(go.Scatter(
    x=[min(y_pred), max(y_pred)],
    y=[0, 0],
    mode='lines',
    line=dict(color='red', dash='dash'),
    name='Zéro'
))

# Layout
fig.update_layout(
    title="Residual Plot",
    xaxis_title="Predictions",
    yaxis_title="Residuals",
    showlegend=True
)

fig.show()


In [16]:
# Création du DataFrame pour Plotly Express
df_plot = pd.DataFrame({
    'y_test': y_test,
    'y_pred': y_pred
})

# Scatter plot
fig = px.scatter(
    df_plot,
    x='y_test',
    y='y_pred',
    labels={'y_test': 'Valeurs réelles tch', 'y_pred': 'Prédictions'},
    title='Prédictions vs Valeurs réelles'
)

# Ajouter une ligne y=x pour visualiser l’idéal
fig.add_shape(
    type='line',
    x0=df_plot['y_test'].min(),
    y0=df_plot['y_test'].min(),
    x1=df_plot['y_test'].max(),
    y1=df_plot['y_test'].max(),
    line=dict(color='red', dash='dash')
)

fig.show()
